In [1]:
!pip install -q segmentation-models-pytorch rasterio albumentations opencv-python

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 154.8/154.8 kB 4.1 MB/s eta 0:00:00


In [2]:
import os
import numpy as np
import rasterio
import torch
import pandas as pd
import cv2
from torch.utils.data import Dataset, DataLoader
import segmentation_models_pytorch as smp
from tqdm import tqdm

In [3]:
BASE_PATH = "/kaggle/input/competitions/anrfaisehack-theme-1-phase2/data"

IMG_DIR   = os.path.join(BASE_PATH, "image")
LABEL_DIR = os.path.join(BASE_PATH, "label")

TRAIN_TXT = os.path.join(BASE_PATH, "split", "train.txt")
VAL_TXT   = os.path.join(BASE_PATH, "split", "val.txt")

PRED_DIR = os.path.join(BASE_PATH, "prediction", "image")

In [4]:
class FloodDataset(Dataset):
    def __init__(self, img_dir, label_dir=None, file_list=None):
        self.img_dir = img_dir
        self.label_dir = label_dir

        with open(file_list) as f:
            self.files = f.read().splitlines()

    def __len__(self):
        return len(self.files)

    def __getitem__(self, idx):
        file_id = self.files[idx]

        img_path = os.path.join(self.img_dir, file_id + "_image.tif")
        with rasterio.open(img_path) as src:
            img = src.read().astype(np.float32)

        img = (img - img.mean()) / (img.std() + 1e-6)
        img = torch.tensor(img)

        if self.label_dir:
            label_path = os.path.join(self.label_dir, file_id + "_label.tif")
            with rasterio.open(label_path) as src:
                mask = src.read(1)

            mask = (mask == 1).astype(np.uint8)   
            mask = torch.tensor(mask).long()
            return img, mask

        return img, file_id

In [5]:
class TestDataset(Dataset):
    def __init__(self, img_dir):
        self.img_dir = img_dir

        self.files = [
            f.replace("_image.tif", "")
            for f in os.listdir(img_dir)
            if f.endswith("_image.tif")
        ]

    def __len__(self):
        return len(self.files)

    def __getitem__(self, idx):
        file_id = self.files[idx]

        path = os.path.join(self.img_dir, file_id + "_image.tif")

        with rasterio.open(path) as src:
            img = src.read().astype(np.float32)

        img = (img - img.mean()) / (img.std() + 1e-6)
        img = torch.tensor(img)

        return img, file_id

In [6]:
train_ds = FloodDataset(IMG_DIR, LABEL_DIR, TRAIN_TXT)
val_ds   = FloodDataset(IMG_DIR, LABEL_DIR, VAL_TXT)

test_ds  = TestDataset(PRED_DIR)

print("Train:", len(train_ds))
print("Val:", len(val_ds))
print("Test:", len(test_ds))  

train_loader = DataLoader(train_ds, batch_size=4, shuffle=True)
val_loader   = DataLoader(val_ds, batch_size=4)
test_loader  = DataLoader(test_ds, batch_size=1)

Train: 59
Val: 10
Test: 19


In [7]:
device = "cuda" if torch.cuda.is_available() else "cpu"

model = smp.Unet(
    encoder_name="efficientnet-b4",
    encoder_weights="imagenet",
    in_channels=6,
    classes=2
).to(device)

config.json:   0%|          | 0.00/106 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/77.9M [00:00<?, ?B/s]

In [8]:
dice_loss = smp.losses.DiceLoss(mode='multiclass')
focal_loss = smp.losses.FocalLoss(mode='multiclass')

optimizer = torch.optim.AdamW(model.parameters(), lr=1e-4)

In [9]:
print("Training Started")

for epoch in range(30):
    model.train()
    total_loss = 0

    for imgs, masks in tqdm(train_loader):
        imgs, masks = imgs.to(device), masks.to(device)

        out = model(imgs)
        loss = dice_loss(out, masks) + focal_loss(out, masks)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    print(f"Epoch {epoch+1} Loss: {total_loss:.4f}")

torch.save(model.state_dict(), "model.pth")
print("Model Saved")

Training Started


100%|██████████| 15/15 [00:16<00:00,  1.07s/it]


Epoch 1 Loss: 15.0528


100%|██████████| 15/15 [00:09<00:00,  1.57it/s]


Epoch 2 Loss: 13.5271


...

Epoch 28 Loss: 6.4798


100%|██████████| 15/15 [00:10<00:00,  1.47it/s]


Epoch 29 Loss: 6.5433


100%|██████████| 15/15 [00:10<00:00,  1.47it/s]

Epoch 30 Loss: 6.4888
Model Saved


In [10]:
def rle_encode(mask):
    pixels = mask.flatten(order='F')
    pixels = np.concatenate([[0], pixels, [0]])
    runs = np.where(pixels[1:] != pixels[:-1])[0] + 1
    runs[1::2] -= runs[::2]
    return " ".join(map(str, runs))

In [11]:
def tta_predict(img):
    preds = []

    preds.append(model(img))

    preds.append(torch.flip(model(torch.flip(img, [3])), [3]))
    preds.append(torch.flip(model(torch.flip(img, [2])), [2]))

    return torch.mean(torch.stack(preds), dim=0)

In [12]:
model.eval()
results = []

print(" Predicting...")

with torch.no_grad():
    for imgs, fid in tqdm(test_loader):
        imgs = imgs.to(device)

        out = tta_predict(imgs)

        prob = torch.softmax(out, dim=1)[:,1,:,:]

        
        flood = (prob > 0.4).cpu().numpy()[0].astype(np.uint8)

        
        flood = cv2.medianBlur(flood, 5)

        rle = "0 0" if flood.sum() == 0 else rle_encode(flood)
        results.append([fid[0], rle])

 Predicting...


100%|██████████| 19/19 [00:04<00:00,  4.30it/s]


In [13]:
df = pd.DataFrame(results, columns=["id", "rle_mask"])
df.to_csv("submission.csv", index=False)

print("submission.csv created!")
print(df.shape)  

submission.csv created!
(19, 2)
